In [2]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle

In [3]:
!kaggle datasets download -d salader/dogsVScats

Dataset URL: https://www.kaggle.com/datasets/salader/dogsVScats
License(s): unknown
 93% 0.99G/1.06G [00:07<00:01, 44.9MB/s]
100% 1.06G/1.06G [00:07<00:00, 150MB/s] 


In [4]:
import zipfile
zip_ref = zipfile.ZipFile('/content/dogsVScats.zip','r')
zip_ref.extractall('/content')
zip_ref.close()

In [23]:
import numpy as np

import tensorflow
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras import Sequential
from keras.layers import Dense,Conv2D,Flatten,MaxPool2D,BatchNormalization,Dropout

In [15]:
batch_size = 32
train_datagen = ImageDataGenerator(
    zoom_range=0.3,
    rotation_range = 0.2,
    height_shift_range = 0.3,
    shear_range=0.3,
    vertical_flip = True,
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_ds = train_datagen.flow_from_directory(
    '/content/train',
    class_mode='binary',
    batch_size=batch_size,
    target_size=(150,150)
)

validation_ds = test_datagen.flow_from_directory(
    '/content/test',
    batch_size=batch_size,
    target_size=(150,150),
    class_mode='binary'
)


Found 20000 images belonging to 2 classes.
Found 5000 images belonging to 2 classes.


In [25]:
model = Sequential()

model.add(Conv2D(32, kernel_size=(3,3), padding='valid', activation='relu',input_shape=(150,150,3)))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(64, kernel_size=(3,3), padding='valid', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Conv2D(128, kernel_size=(3,3), padding='valid', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2,2),strides=2,padding='valid'))

model.add(Flatten())

model.add(Dense(32, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(1, activation='sigmoid'))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_13 (Conv2D)              │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 148, 148, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 72, 72, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 34, 34, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 36992)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 32)             │     1,183,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,279,009 (4.88 MB)

 Trainable params: 1,278,561 (4.88 MB)

 Non-trainable params: 448 (1.75 KB)

In [26]:
model.compile(optimizer='rmsprop', loss='binary_crossentropy',metrics=['accuracy'])

In [27]:
history = model.fit(
    train_ds,
    steps_per_epoch = int(2000 / batch_size),
    epochs=25,
    validation_data = validation_ds,
    validation_steps = int(800 / batch_size)
)

Epoch 1/25
62/62 ━━━━━━━━━━━━━━━━━━━━ 21s 247ms/step - accuracy: 0.5254 - loss: 2.1783 - val_accuracy: 0.5300 - val_loss: 0.7297
Epoch 2/25
62/62 ━━━━━━━━━━━━━━━━━━━━ 15s 238ms/step - accuracy: 0.5157 - loss: 1.0492 - val_accuracy: 0.4787 - val_loss: 1.0041
Epoch 3/25
62/62 ━━━━━━━━━━━━━━━━━━━━ 15s 240ms/step - accuracy: 0.5496 - loss: 0.7779 - val_accuracy: 0.5163 - val_loss: 0.7540
Epoch 4/25
62/62 ━━━━━━━━━━━━━━━━━━━━ 15s 241ms/step - accuracy: 0.5423 - loss: 0.8344 - val_accuracy: 0.5100 - val_loss: 0.8251
Epoch 5/25
62/62 ━━━━━━━━━━━━━━━━━━━━ 14s 234ms/step - accuracy: 0.5442 - loss: 0.7289 - val_accuracy: 0.4925 - val_loss: 1.6117
Epoch 6/25
62/62 ━━━━━━━━━━━━━━━━━━━━ 14s 230ms/step - accuracy: 0.5549 - loss: 0.7206 - val_accuracy: 0.4800 - val_loss: 1.6291
Epoch 7/25
62/62 ━━━━━━━━━━━━━━━━━━━━ 14s 231ms/step - accuracy: 0.5154 - loss: 0.7201 - val_accuracy: 0.4888 - val_loss: 0.7981
Epoch 8/25
62/62 ━━━━━━━━━━━━━━━━━━━━ 14s 231ms/step - accuracy: 0.4984 - loss: 0.7233 - val_accu

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.5731 - loss: 0.6973 - val_accuracy: 0.4925 - val_loss: 0.8009
Epoch 12/25
62/62 ━━━━━━━━━━━━━━━━━━━━ 15s 240ms/step - accuracy: 0.5452 - loss: 0.6865 - val_accuracy: 0.5038 - val_loss: 0.8176
Epoch 13/25
62/62 ━━━━━━━━━━━━━━━━━━━━ 15s 241ms/step - accuracy: 0.5080 - loss: 0.7040 - val_accuracy: 0.5025 - val_loss: 0.7212
Epoch 14/25
62/62 ━━━━━━━━━━━━━━━━━━━━ 14s 233ms/step - accuracy: 0.5414 - loss: 0.6871 - val_accuracy: 0.5225 - val_loss: 0.7020
Epoch 15/25
62/62 ━━━━━━━━━━━━━━━━━━━━ 14s 231ms/step - accuracy: 0.5975 - loss: 0.6709 - val_accuracy: 0.4450 - val_loss: 0.6941
Epoch 16/25
62/62 ━━━━━━━━━━━━━━━━━━━━ 16s 257ms/step - accuracy: 0.5718 - loss: 0.7084 - val_accuracy: 0.4825 - val_loss: 2.9381
Epoch 17/25
62/62 ━━━━━━━━━━━━━━━━━━━━ 14s 234ms/step - accuracy: 0.5926 - loss: 0.6822 - val_accuracy: 0.5188 - val_loss: 0.6925
Epoch 18/25
62/62 ━━━━━━━━━━━━━━━━━━━━ 14s 232ms/step - accuracy: 0.5601 - loss: 0.6890 - val_accuracy: 

# The purpose is not to create best model but to practice the code of Data augmentation